[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/Likelihood_Error_Analyzer_Multi_Models_Enhanced_v3.ipynb)

**📝 Before using:** Update the GitHub URL above with your actual username and repository name.

# Likelihood Error Analyzer - Enhanced Multi-Model v3

## What's New in v3

✨ **Enhanced LLM Judge with Multiple Model Support:**
- Support for Claude Sonnet 4.5, GPT-4o, GPT-4o-mini, and open source models
- Improved prompt with better scoring logic understanding
- Deterministic validation checks before LLM review
- Post-processing quality validation
- Easy model switching via configuration

## Overview

**Phase 1 (Always runs):** Computes deterministic Likelihood of Error Score (0–5) for ALL records

**Phase 2 (Optional):** Uses LLM to evaluate ONLY Moderate, High, and Very High risk records

### Required Inputs
- `Job_Classifications_Batch.json` (or .csv)
- `alternative_roles_analysis.json` (or .csv)
- `Role_Confusion_Crosswalk.json` (or .csv)
- `Universal_Role_Classification_Prompt.json` (or .txt)

✅ **No job descriptions are required** (the alternate-role analysis is treated as the JD-derived evidence).


## 🚀 Quick Start Guide

### Option 1: Deterministic Only (Fast, No API Keys Required)
Run cells **1-9** only:
1. Install dependencies (Cell 1)
2. Upload your 4 input files (Cell 2)
3. Cells 3-9 will automatically compute likelihood scores
4. Skip to Cell 14 to export results

**Time:** ~2-3 minutes for 63 records

---

### Option 2: Hybrid Approach (Recommended for Production)
Run cells **1-14** sequentially:
1. Complete deterministic scoring (Cells 1-9)
2. Configure your AI model (Cell 10) - API key from Secrets
3. Run LLM evaluation on **Moderate+ risk records only** (Cell 11-12)
4. Export enhanced results (Cell 14)

**Time:** ~5-10 minutes (depending on # of Moderate+ records and model)

**Cost:** Typically evaluates only 10-20% of records with LLM

---

### Supported Models

**Tier 1 - Production (Recommended):**
- `claude-sonnet-4-20250514` (Claude Sonnet 4.5) - Best accuracy
- `gpt-4o` (GPT-4o latest) - Excellent accuracy
- `gpt-4o-mini` (GPT-4o-mini) - Best budget option

**Tier 2 - Open Source:**
- `Qwen/Qwen2.5-72B-Instruct` (HuggingFace)
- `Qwen/Qwen2.5-7B-Instruct` (HuggingFace) - Not recommended

**API Keys Required:**
- OpenAI models: Store `OPENAI_API_KEY` in Colab Secrets
- Claude models: Store `ANTHROPIC_API_KEY` in Colab Secrets  
- HuggingFace models: Store `HF_TOKEN` in Colab Secrets


In [19]:
# ==== 0) Install dependencies (Colab) ====
!pip -q install pandas numpy matplotlib anthropic openai huggingface_hub tqdm


In [20]:
 # NEW ==== 1) Upload inputs (.csv to convert to JSON files) ====
import pandas as pd
import json
from google.colab import files

# 1. Define the data mapping
conversions = {
    'Job Classifications Batch.csv': 'Job_Classifications_Batch.json',
    'alternative_roles_analysis.csv': 'alternative_roles_analysis.json',
    'Role_Confusion_Crosswalk.csv': 'Role_Confusion_Crosswalk.json'
}

# 2. Process CSV to JSON
for csv_fn, json_fn in conversions.items():
    try:
        df = pd.read_csv(csv_fn)
        df.to_json(json_fn, orient='records', indent=2)
        print(f"✅ Created {json_fn}")
        files.download(json_fn)
    except Exception as e:
        print(f"❌ Error processing {csv_fn}: {e}")

# 3. Process the Text Prompt to JSON
try:
    with open('Universal_Role_Classification_Prompt.txt', 'r') as f:
        content = f.read()

    prompt_json = {
        "file_name": "Universal_Role_Classification_Prompt.txt",
        "file_type": "text/plain",
        "content": content,
        "metadata": {}
    }

    with open("Universal_Role_Classification_Prompt.json", "w") as f:
        json.dump(prompt_json, f, indent=2)

    print("✅ Created Universal_Role_Classification_Prompt.json")
    files.download("Universal_Role_Classification_Prompt.json")
except Exception as e:
    print(f"❌ Error processing Prompt TXT: {e}")

❌ Error processing Job Classifications Batch.csv: [Errno 2] No such file or directory: 'Job Classifications Batch.csv'
❌ Error processing alternative_roles_analysis.csv: [Errno 2] No such file or directory: 'alternative_roles_analysis.csv'
❌ Error processing Role_Confusion_Crosswalk.csv: [Errno 2] No such file or directory: 'Role_Confusion_Crosswalk.csv'
❌ Error processing Prompt TXT: [Errno 2] No such file or directory: 'Universal_Role_Classification_Prompt.txt'


In [ ]:
# ==== 1) Upload inputs (ZIP or individual JSON files) ====
from google.colab import files
import os, zipfile, glob

uploaded = files.upload()

ZIP_NAME = "Likelihood Evaluation Resources.zip"
WORKDIR = "/content/likelihood_eval"
os.makedirs(WORKDIR, exist_ok=True)

# If ZIP uploaded, extract it into WORKDIR
if ZIP_NAME in uploaded:
    zip_path = os.path.join("/content", ZIP_NAME)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(WORKDIR)
    print(f"✅ Extracted {ZIP_NAME} to {WORKDIR}")

# Move any individually uploaded files into WORKDIR
for fn in uploaded.keys():
    src = os.path.join("/content", fn)
    dst = os.path.join(WORKDIR, fn)
    if os.path.exists(src) and src != dst:
        os.replace(src, dst)

print(f"✅ Working directory: {WORKDIR}")
print("Files found:", [os.path.basename(p) for p in glob.glob(os.path.join(WORKDIR, '*'))])

def find_file(candidates):
    cand_lower = [c.lower() for c in candidates]
    for c in candidates:
        p = os.path.join(WORKDIR, c)
        if os.path.exists(p):
            return p
    for p in glob.glob(os.path.join(WORKDIR, "*")):
        if os.path.basename(p).lower() in cand_lower:
            return p
    raise FileNotFoundError(f"Could not find any of: {candidates} in {WORKDIR}")

PATH_ALT       = find_file(["alternative_roles_analysis.json"])
PATH_JOB_BATCH = find_file(["Job_Classifications_Batch.json"])
PATH_CROSSWALK = find_file(["Role_Confusion_Crosswalk.json"])
PATH_PROMPT    = find_file(["Universal_Role_Classification_Prompt.json"])

print("✅ Using:")
print(" - alternative_roles_analysis:", PATH_ALT)
print(" - Job_Classifications_Batch :", PATH_JOB_BATCH)
print(" - Role_Confusion_Crosswalk  :", PATH_CROSSWALK)
print(" - Universal prompt          :", PATH_PROMPT)


In [ ]:
# ==== 2) Load JSON files into DataFrames ====
import json
import pandas as pd
import numpy as np

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

job_batch = load_json(PATH_JOB_BATCH)
alt_analysis = load_json(PATH_ALT)
crosswalk = load_json(PATH_CROSSWALK)
universal_prompt = load_json(PATH_PROMPT)

df_jobs = pd.DataFrame(job_batch if isinstance(job_batch, list) else job_batch.get("rows", []))
df_alt  = pd.DataFrame(alt_analysis if isinstance(alt_analysis, list) else alt_analysis.get("rows", []))
df_cross = pd.DataFrame(crosswalk if isinstance(crosswalk, list) else crosswalk.get("rows", []))

print("df_jobs :", df_jobs.shape)
print("df_alt  :", df_alt.shape)
print("df_cross:", df_cross.shape)

display(df_jobs.head(3))


In [ ]:
# NEW ==== 3) Key fields + safe normalization ====
import re

def norm(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    # Standardize to lowercase and remove leading/trailing spaces for reliable matching
    return str(s).strip().lower()

# Identify columns in the batch file
title_cols = [c for c in df_jobs.columns if c.lower() in ["job_title_original", "job_title", "title"]]
role_cols  = [c for c in df_jobs.columns if c.lower() in ["major_role_group", "major_role", "role"]]

if not title_cols or not role_cols:
    raise KeyError(f"Missing required columns. Found: {list(df_jobs.columns)}")

TITLE_COL = title_cols[0]
ROLE_COL  = role_cols[0]

# Standardize the 'Major Role Group' as the primary join key
df_jobs["role_key"] = df_jobs[ROLE_COL].map(norm)
df_jobs["job_title_key"] = df_jobs[TITLE_COL].map(norm)

# Standardize Alternative Analysis keys
# Using 'Classified Role' as the key to match against the batch's major_role_group
alt_key_cols = [c for c in df_alt.columns if c.lower() in ["classified role", "role", "major_role_group"]]
if alt_key_cols:
    df_alt["role_key"] = df_alt[alt_key_cols[0]].map(norm)

# Standardize Crosswalk keys
# Using 'Role' as the key to match against the batch's major_role_group
cross_key_cols = [c for c in df_cross.columns if c.lower() in ["role", "role_a", "job_title"]]
if cross_key_cols:
    df_cross["role_key"] = df_cross[cross_key_cols[0]].map(norm)

print("✅ Normalization complete. Data is now linked via 'role_key'.")

In [ ]:
# ==== 3) Key fields + safe normalization ====
import re

def norm(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    return str(s).strip()

# Job batch: find title + major role group
title_cols = [c for c in df_jobs.columns if c.lower() in ["job_title_original","job title","job_title","title","new_job_title"]]
role_cols  = [c for c in df_jobs.columns if c.lower() in ["major_role_group","major role group","major_role","major"]]

if not title_cols or not role_cols:
    raise KeyError(f"Could not find job title / major role columns. Columns found: {list(df_jobs.columns)}")

TITLE_COL = title_cols[0]
ROLE_COL  = role_cols[0]

df_jobs["job_title_key"] = df_jobs[TITLE_COL].map(norm)
df_jobs["major_role_group"] = df_jobs[ROLE_COL].map(norm)

# Alternative analysis: prefer Job Code linkage if present, else title
ALT_JOB_CODE_COL = None
for c in df_alt.columns:
    if c.lower().replace(" ", "") in ["jobcode","job_code","jobcodenumber"]:
        ALT_JOB_CODE_COL = c
        break

ALT_TITLE_COL = None
for c in df_alt.columns:
    if c.lower() in ["job_title_original","job title","job_title","title","job_title_key"]:
        ALT_TITLE_COL = c
        break

if ALT_JOB_CODE_COL:
    print(f"Using Job Code for alt linkage: {ALT_JOB_CODE_COL}")
    df_alt["job_title_key"] = df_alt[ALT_JOB_CODE_COL].map(norm)
elif ALT_TITLE_COL:
    print(f"Using Title for alt linkage: {ALT_TITLE_COL}")
    df_alt["job_title_key"] = df_alt[ALT_TITLE_COL].map(norm)
else:
    raise KeyError(f"Could not find job code or title column in alt_analysis. Columns: {list(df_alt.columns)}")

# Crosswalk linkage
CROSS_TITLE_COL = None
for c in df_cross.columns:
    if c.lower() in ["job_title_original","job title","job_title","title","role_a"]:
        CROSS_TITLE_COL = c
        break

if CROSS_TITLE_COL:
    df_cross["job_title_key"] = df_cross[CROSS_TITLE_COL].map(norm)
else:
    print("⚠️ Warning: Could not find title column in crosswalk. Crosswalk signals may be unavailable.")
    df_cross["job_title_key"] = ""

print("✅ Normalization complete")


In [ ]:
# ==== 4) Baseline human error probabilities ====
# Based on cognitive psychology research and HR classification complexity

BASELINE_ERROR_PROBS = {
    "Teacher": 18,
    "Manager": 24,
    "Analyst": 24,
    "Director": 18,
    "Coordinator": 24,
    "Specialist": 30,
    "Technician": 22,
    "Assistant": 20,
    "Officer": 22,
    "Designer": 18,
    "Facilitator": 26,
    "Instructor": 18,
    "Tutor": 20,
    "Engineer": 20,
    "Developer": 20,
    "Administrator": 24,
    "Secretary": 16,
    "Clerk": 16,
    "Principal": 18,
    "Supervisor": 22,
    "Chief": 18,
    "Associate": 26,
    "Consultant": 28,
    "Counselor": 20,
    "Therapist": 20,
    "Nurse": 18,
    "Auditor": 22,
    "Investigator": 24,
    "Accountant": 20,
    "Executive Officer": 18,
    "Mechanic": 18,
    "Operator": 18,
    "Driver": 18,
}

DEFAULT_ERROR_PROB = 24  # For any role not in the dictionary

def get_baseline_error(role):
    """Get baseline human error probability for a role"""
    role_normalized = norm(role)
    return BASELINE_ERROR_PROBS.get(role_normalized, DEFAULT_ERROR_PROB)

df_jobs["human_error_probability"] = df_jobs["major_role_group"].apply(get_baseline_error)

print("✅ Baseline error probabilities assigned")
print(f"Average baseline error: {df_jobs['human_error_probability'].mean():.1f}%")


In [ ]:
# NEW ==== 5) Merge alternative roles analysis ====

def extract_alt_roles(row):
    """Extract list of alternative roles from your specific JSON structure"""
    # Your file uses 'Other Plausible Roles'
    target_cols = ["Other Plausible Roles", "suggested_alternative_roles", "alternative_roles"]
    for col in target_cols:
        if col in row and row[col]:
            val = row[col]
            if isinstance(val, list): return val
            if isinstance(val, str):
                # Handle comma-separated strings like "Instructor, Tutor"
                return [v.strip() for v in val.split(",") if v.strip()]
    return []

# Apply extraction
df_alt["alt_roles_list"] = df_alt.apply(extract_alt_roles, axis=1)
df_alt["alt_count"] = df_alt["alt_roles_list"].apply(len)

# Since there might be multiple job codes for one role, we take the most detailed analysis
alt_lookup = df_alt.groupby("role_key").agg({
    "alt_roles_list": "first",
    "alt_count": "max"
}).reset_index()

# Merge into main dataframe using the normalized role_key
df_jobs = df_jobs.merge(alt_lookup, on="role_key", how="left")

# Clean up missing values
df_jobs["alt_roles_list"] = df_jobs["alt_roles_list"].apply(lambda x: x if isinstance(x, list) else [])
df_jobs["alt_count"] = df_jobs["alt_count"].fillna(0).astype(int)
df_jobs["pattern_hit"] = 0 # Default if not provided

print(f"✅ Alternative roles merged. Found evidence for {len(alt_lookup)} role types.")

In [ ]:
# ==== 5) Merge alternative roles analysis ====

# Extract alternative roles list from df_alt
def extract_alt_roles(row):
    """Extract list of alternative roles from various possible column formats"""
    # Check for direct list column
    for col in ["alternative_roles", "alt_roles", "alternatives", "other_roles"]:
        if col in row and row[col]:
            val = row[col]
            if isinstance(val, list):
                return val
            if isinstance(val, str):
                try:
                    return json.loads(val)
                except:
                    return [v.strip() for v in val.split(",") if v.strip()]
    return []

df_alt["alt_roles_list"] = df_alt.apply(extract_alt_roles, axis=1)
df_alt["alt_count"] = df_alt["alt_roles_list"].apply(len)

# Check for pattern_hit column
if "pattern_hit" not in df_alt.columns:
    df_alt["pattern_hit"] = 0

# Merge into main dataframe
alt_merge_cols = ["job_title_key", "alt_roles_list", "alt_count", "pattern_hit"]
available_alt_cols = [c for c in alt_merge_cols if c in df_alt.columns]

df_jobs = df_jobs.merge(
    df_alt[available_alt_cols],
    on="job_title_key",
    how="left"
)

# Fill missing values
df_jobs["alt_roles_list"] = df_jobs["alt_roles_list"].apply(lambda x: x if isinstance(x, list) else [])
df_jobs["alt_count"] = df_jobs["alt_count"].fillna(0).astype(int)
df_jobs["pattern_hit"] = df_jobs["pattern_hit"].fillna(0).astype(int)

print("✅ Alternative roles merged")
print(f"Records with alternatives: {(df_jobs['alt_count'] > 0).sum()}")
print(f"Average alternatives per record: {df_jobs['alt_count'].mean():.2f}")


In [ ]:
# NEW ==== 6) Merge crosswalk confusion signals ====

if "role_key" in df_cross.columns:
    # Use actual column names from your Crosswalk file
    crosswalk_agg = df_cross.groupby("role_key").agg({
        "Confusion Risk Score": "max",
        "Top Match Role": "first"
    }).reset_index().rename(columns={
        "Confusion Risk Score": "confusion_risk_score",
        "Top Match Role": "top_match_role"
    })

    # Merge into main dataframe
    df_jobs = df_jobs.merge(crosswalk_agg, on="role_key", how="left")
    df_jobs["crosswalk_confirmed"] = df_jobs["confusion_risk_score"].notna().astype(int)

    print(f"✅ Crosswalk data merged for {len(crosswalk_agg)} roles.")
else:
    print("⚠️ No crosswalk data matched.")
    df_jobs["confusion_risk_score"] = 0
    df_jobs["top_match_role"] = "Any role"
    df_jobs["crosswalk_confirmed"] = 0

# Fill NaNs from merge
df_jobs["confusion_risk_score"] = df_jobs["confusion_risk_score"].fillna(0).astype(int)
df_jobs["top_match_role"] = df_jobs["top_match_role"].fillna("Any role")

In [ ]:
# ==== 6) Merge crosswalk confusion signals ====

# Aggregate crosswalk data by job title
if "job_title_key" in df_cross.columns and len(df_cross) > 0:
    crosswalk_agg = df_cross.groupby("job_title_key").agg({
        "overall_confusion_risk": "max",  # Highest confusion risk for this title
    }).reset_index()

    # Find most likely misclassification (role with highest confusion)
    def get_top_confused_role(title):
        matches = df_cross[df_cross["job_title_key"] == title]
        if len(matches) == 0:
            return "Any role"
        if "role_b" in matches.columns and "overall_confusion_risk" in matches.columns:
            top_match = matches.loc[matches["overall_confusion_risk"].idxmax()]
            return norm(top_match.get("role_b", "Any role"))
        return "Any role"

    crosswalk_agg["top_match_role"] = crosswalk_agg["job_title_key"].apply(get_top_confused_role)
    crosswalk_agg["crosswalk_confirmed"] = 1

    # Merge into main dataframe
    df_jobs = df_jobs.merge(
        crosswalk_agg,
        on="job_title_key",
        how="left"
    )

    print("✅ Crosswalk data merged")
    print(f"Records with crosswalk signals: {df_jobs['crosswalk_confirmed'].sum()}")
else:
    print("⚠️ No crosswalk data available")
    df_jobs["overall_confusion_risk"] = 0
    df_jobs["top_match_role"] = "Any role"
    df_jobs["crosswalk_confirmed"] = 0

# Standardize column name
df_jobs["confusion_risk_score"] = df_jobs["overall_confusion_risk"].fillna(0).astype(int)
df_jobs["crosswalk_confirmed"] = df_jobs["crosswalk_confirmed"].fillna(0).astype(int)


In [ ]:
# ==== 7) Determine most likely misclassification ====

def determine_most_likely_misclass(row):
    """Determine most likely misclassification based on multiple signals"""

    # Priority 1: Crosswalk top match if confusion risk > 0
    if row.get("confusion_risk_score", 0) > 0 and row.get("top_match_role", "") != "Any role":
        return row["top_match_role"]

    # Priority 2: First alternative role if available
    alt_roles = row.get("alt_roles_list", [])
    if isinstance(alt_roles, list) and len(alt_roles) > 0:
        return alt_roles[0]

    # Priority 3: Pattern match suggests multiple possibilities
    if row.get("pattern_hit", 0) > 0:
        return "Multiple possibilities"

    # Default: Any role (low specificity)
    return "Any role"

df_jobs["most_likely_misclassification"] = df_jobs.apply(determine_most_likely_misclass, axis=1)

print("✅ Most likely misclassifications determined")
print(f"Specific misclassification identified: {(df_jobs['most_likely_misclassification'] != 'Any role').sum()}")


In [ ]:
# ==== 8) Compute likelihood error score (0-5 scale) ====

def compute_likelihood_score(row):
    """
    Compute likelihood of error score based on:
    1. Baseline human error probability for role type
    2. Number of alternative roles (indicates ambiguity)
    3. Crosswalk confusion signals
    4. Pattern matching hits

    Returns: float between 0-5
    """

    # Start with baseline (convert percentage to 0-5 scale: 24% -> ~1.2)
    base_score = row.get("human_error_probability", 24) / 20.0

    # Add for alternative roles (each adds uncertainty)
    alt_count = row.get("alt_count", 0)
    alt_penalty = min(alt_count * 0.5, 2.0)  # Cap at +2.0

    # Add for crosswalk confusion
    confusion_score = row.get("confusion_risk_score", 0)
    confusion_penalty = min(confusion_score * 0.5, 1.5)  # Cap at +1.5

    # Small boost if pattern matching flagged this
    pattern_boost = 0.2 if row.get("pattern_hit", 0) > 0 else 0

    # Total score (cap at 5.0)
    total = base_score + alt_penalty + confusion_penalty + pattern_boost
    return min(round(total, 1), 5.0)

df_jobs["p_error"] = df_jobs["human_error_probability"] / 100.0
df_jobs["likelihood_error_score_0_5"] = df_jobs.apply(compute_likelihood_score, axis=1)

# Assign likelihood bands
def assign_band(score):
    if score < 1.0:
        return "Very Low"
    elif score < 2.0:
        return "Low"
    elif score < 3.0:
        return "Moderate"
    elif score < 4.0:
        return "High"
    else:
        return "Very High"

df_jobs["likelihood_band"] = df_jobs["likelihood_error_score_0_5"].apply(assign_band)

print("✅ Likelihood scores computed")
print("\n📊 Score Distribution:")
print(df_jobs["likelihood_band"].value_counts().sort_index())
print(f"\nAverage score: {df_jobs['likelihood_error_score_0_5'].mean():.2f}")


In [ ]:
# ==== 9) Create consolidated dataframe for export ====

# Select and order columns for final output
output_cols = [
    "job_title_key",
    "major_role_group",
    "human_error_probability",
    "confusion_risk_score",
    "most_likely_misclassification",
    "top_match_role",
    "alt_roles_list",
    "alt_count",
    "pattern_hit",
    "crosswalk_confirmed",
    "p_error",
    "likelihood_error_score_0_5",
    "likelihood_band",
]

# Add source columns if they exist
for col in ["source_row_index", "job_title_original", "new_job_title", "minor_sub_group",
            "grouping_justification", "model_used", "reason_pass5"]:
    if col in df_jobs.columns:
        output_cols.insert(0, col)

# Remove duplicates
output_cols = list(dict.fromkeys(output_cols))
available_cols = [c for c in output_cols if c in df_jobs.columns]

df = df_jobs[available_cols].copy()

print("✅ Base dataframe created")
print(f"Total records: {len(df)}")
print(f"\n📊 Records by Risk Band:")
print(df["likelihood_band"].value_counts())

# Display sample
print("\n📋 Sample Records:")
display_cols = ["job_title_key", "major_role_group", "likelihood_error_score_0_5",
                "likelihood_band", "alt_count", "confusion_risk_score"]
display_cols = [c for c in display_cols if c in df.columns]
display(df[display_cols].head(10))


---
## 🤖 LLM Judge Evaluation (Optional)

The cells below use an LLM to evaluate the quality of classifications for **Moderate, High, and Very High** risk records only.

**Supported Models:**
- **Claude Sonnet 4.5** (Recommended - Best accuracy)
- **GPT-4o** (Excellent accuracy)
- **GPT-4o-mini** (Best budget option)
- **Qwen2.5-72B** (Open source)

**Setup:**
1. Choose your model in Cell 10
2. Ensure API key is in Colab Secrets (OPENAI_API_KEY, ANTHROPIC_API_KEY, or HF_TOKEN)
3. Run Cells 10-12


In [ ]:
# NEW ==== 10) Deterministic validation checks (runs before LLM) ====

def validate_score_logic(row):
    score = row['likelihood_error_score_0_5']
    alt_count = row.get('alt_count', 0)
    confusion = row.get('confusion_risk_score', 0)

    # Flags if the score seems inconsistent with the evidence
    if alt_count >= 2 and confusion >= 2 and score < 2.0:
        return "too_low"
    if alt_count == 0 and confusion == 0 and score >= 2.5:
        return "too_high"
    return "appropriate"

# Inclusion of 'Low' ensures the LLM processes a wider variety of records
RISK_BANDS_FOR_REVIEW = ["Low", "Moderate", "High", "Very High"]
moderate_or_higher = df[df["likelihood_band"].isin(RISK_BANDS_FOR_REVIEW)].copy()

if len(moderate_or_higher) > 0:
    moderate_or_higher['deterministic_check'] = moderate_or_higher.apply(validate_score_logic, axis=1)
    print(f"✅ Prepared {len(moderate_or_higher)} records for LLM Review.")
    print(moderate_or_higher['likelihood_band'].value_counts())
else:
    print("⚠️ No records matched the review criteria. Check normalization in Cell 3.")

In [ ]:
# ==== 10) Deterministic validation checks (runs before LLM) ====

def validate_score_logic(row):
    """
    Check if score aligns with evidence using deterministic rules.
    This catches obvious mismatches before expensive LLM evaluation.
    """
    score = row['likelihood_error_score_0_5']
    alt_count = row.get('alt_count', 0)
    confusion = row.get('confusion_risk_score', 0)

    # Rule 1: Multiple alternatives + confusion should mean score >= 2.0
    if alt_count >= 2 and confusion >= 2 and score < 2.0:
        return "too_low"

    # Rule 2: No alternatives and no confusion should mean score < 2.0
    if alt_count == 0 and confusion == 0 and score >= 2.5:
        return "too_high"

    # Rule 3: Very high alt_count should mean higher score
    if alt_count >= 4 and score < 2.5:
        return "too_low"

    return "appropriate"

# Apply to all Moderate+ records
RISK_BANDS_FOR_REVIEW = ["Moderate", "High", "Very High"]
moderate_or_higher = df[df["likelihood_band"].isin(RISK_BANDS_FOR_REVIEW)].copy()

if len(moderate_or_higher) > 0:
    moderate_or_higher['deterministic_check'] = moderate_or_higher.apply(validate_score_logic, axis=1)

    # Report findings
    check_counts = moderate_or_higher['deterministic_check'].value_counts()
    print("✅ Deterministic validation complete")
    print(f"\n📊 Pre-LLM Validation Results ({len(moderate_or_higher)} records):")
    print(check_counts)

    if check_counts.get('too_low', 0) > 0 or check_counts.get('too_high', 0) > 0:
        print("\n⚠️ Some scores flagged by deterministic rules - LLM will provide deeper analysis")
else:
    print("✅ No Moderate+ risk records found - LLM evaluation not needed")


In [ ]:
# ==== 11) LLM Judge Configuration ====

from google.colab import userdata
import time

# ========================================
# 🎯 CONFIGURATION - CHANGE MODEL HERE
# ========================================

# Choose your model (uncomment ONE):
MODEL_CHOICE = "gpt-4o-mini"  # Recommended: Best budget option
# MODEL_CHOICE = "claude-sonnet-4.5"  # Best accuracy
# MODEL_CHOICE = "gpt-4o"  # Excellent accuracy
# MODEL_CHOICE = "qwen-72b"  # Open source option

# ========================================
# Model Configuration
# ========================================

MODEL_CONFIGS = {
    "claude-sonnet-4.5": {
        "api_key_name": "ANTHROPIC_API_KEY",
        "model_id": "claude-sonnet-4-20250514",
        "provider": "anthropic",
        "max_tokens": 600,
        "temperature": 0.1,
    },
    "gpt-4o": {
        "api_key_name": "OPENAI_API_KEY",
        "model_id": "gpt-4o",
        "provider": "openai",
        "max_tokens": 600,
        "temperature": 0.1,
    },
    "gpt-4o-mini": {
        "api_key_name": "OPENAI_API_KEY",
        "model_id": "gpt-4o-mini",
        "provider": "openai",
        "max_tokens": 600,
        "temperature": 0.1,
    },
    "qwen-72b": {
        "api_key_name": "HF_TOKEN",
        "model_id": "Qwen/Qwen2.5-72B-Instruct",
        "provider": "huggingface",
        "max_tokens": 500,
        "temperature": 0.1,
    },
}

# Get configuration
if MODEL_CHOICE not in MODEL_CONFIGS:
    raise ValueError(f"Invalid MODEL_CHOICE: {MODEL_CHOICE}. Choose from: {list(MODEL_CONFIGS.keys())}")

config = MODEL_CONFIGS[MODEL_CHOICE]
print(f"✅ Selected model: {MODEL_CHOICE}")
print(f"   Provider: {config['provider']}")
print(f"   Model ID: {config['model_id']}")

# Get API key from Colab Secrets
try:
    API_KEY = userdata.get(config["api_key_name"])
    print(f"✅ API key loaded from Secrets: {config['api_key_name']}")
except Exception as e:
    print(f"❌ ERROR: Could not load {config['api_key_name']} from Colab Secrets")
    print(f"   Please add your API key to Colab Secrets (🔑 icon in left sidebar)")
    raise

# Initialize clients
if config["provider"] == "anthropic":
    from anthropic import Anthropic
    client = Anthropic(api_key=API_KEY)
    print("✅ Anthropic client initialized")

elif config["provider"] == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY)
    print("✅ OpenAI client initialized")

elif config["provider"] == "huggingface":
    from huggingface_hub import InferenceClient
    client = InferenceClient(api_key=API_KEY)
    print("✅ HuggingFace client initialized")


In [ ]:
# ==== 12) Enhanced LLM Judge System ====

import json
import re
from tqdm.auto import tqdm

# ========================================
# Enhanced Judge System Prompt
# ========================================

JUDGE_SYSTEM = """You are an auditing assistant evaluating AI-generated job classification decisions.

CRITICAL SCORING LOGIC:
The likelihood_error_score (0-5 scale) measures RISK of misclassification based on:
- Base human error rate for the role type (varies by role complexity)
- Number of plausible alternative roles (alt_count)
- Crosswalk confusion signals from similar role titles (confusion_risk_score)
- Pattern matching hits from historical data

SCORE FORMULA: base_error/20 + (alt_count × 0.5) + (confusion_risk × 0.5) + pattern_boost

HIGH SCORES (2.0-3.5) ARE APPROPRIATE when:
✓ Multiple alternative roles exist (alt_count ≥ 2)
✓ Crosswalk shows confusion risk (confusion_risk_score > 0)
✓ Justification explicitly discusses and rejects competing roles
✓ Pattern matching flagged similar roles (pattern_hit = 1)

THIS MEANS: A detailed justification that mentions alternatives CONFIRMS that high risk is real.
The AI correctly identified ambiguity - this justifies a HIGHER score, not lower.

LOW SCORES (0-1.5) ARE APPROPRIATE when:
✓ No alternative roles identified (alt_count = 0)
✓ No crosswalk confusion (confusion_risk_score = 0)
✓ Clear, unambiguous fit with single obvious classification

COMMON MISTAKE TO AVOID:
❌ WRONG: "The justification is detailed and thorough, so the high score seems too_high"
✅ RIGHT: "The justification discusses 3 alternatives and explains rejections, confirming score of 2.6 is appropriate"

Your evaluation criteria:
1. Justification quality issues:
   - Weak evidence or circular reasoning
   - Hedge-heavy language without substance
   - Title-only reasoning ("because the title says...")

2. Alignment between justification content and scoring factors:
   - Does mentioning alternatives match alt_count?
   - Are competing roles discussed appropriately?

3. Score appropriateness given numeric evidence:
   - Does score match alt_count + confusion_risk_score?
   - Is the score too high/low relative to the evidence?

EXAMPLES:

Example 1 - APPROPRIATE High Score:
Input: {"score": 2.6, "alt_count": 3, "confusion_risk": 2, "justification": "Role aligns with Manager. While Director was considered due to scope, the position focuses on operational execution rather than strategic planning..."}
Output: {"score_assessment": "appropriate", "confidence": "high", "notes": "Score matches evidence: 3 alternatives + confusion signals justify 2.6"}

Example 2 - APPROPRIATE Low Score:
Input: {"score": 1.2, "alt_count": 0, "confusion_risk": 0, "justification": "Clear Teacher role with standard instructional responsibilities..."}
Output: {"score_assessment": "appropriate", "confidence": "high", "notes": "Low score correct: no alternatives, no confusion"}

Example 3 - TOO HIGH (Weak Justification):
Input: {"score": 2.8, "alt_count": 3, "confusion_risk": 1, "justification": "This is a Coordinator because the title says Coordinator."}
Output: {"score_assessment": "too_high", "confidence": "medium", "notes": "Title-only reasoning; doesn't justify high score despite alternatives"}

Return ONLY valid JSON matching this exact schema:
{
  "hedging_language": boolean,
  "title_only_reasoning": boolean,
  "mentions_competing_role_terms": boolean,
  "score_assessment": "appropriate" | "too_low" | "too_high",
  "confidence": "high" | "medium" | "low",
  "notes": "brief explanation (max 200 chars)"
}"""

# ========================================
# Universal judge function
# ========================================

def judge_record(row: dict, max_retries=3):
    """
    Judge a single record using configured LLM.
    Works with Claude, OpenAI, or HuggingFace models.
    """

    # Build comprehensive payload
    payload = {
        "job_title_original": row.get("job_title_original"),
        "major_role_group": row.get("major_role_group"),
        "minor_sub_group": row.get("minor_sub_group", ""),
        "likelihood_error_score_0_5": row.get("likelihood_error_score_0_5"),
        "likelihood_band": row.get("likelihood_band"),

        # KEY SCORING FACTORS
        "alt_count": row.get("alt_count", 0),
        "confusion_risk_score": row.get("confusion_risk_score", 0),
        "pattern_hit": row.get("pattern_hit", 0),
        "human_error_probability": row.get("human_error_probability", 0),

        # CONTEXT
        "top_match_role": row.get("top_match_role"),
        "alt_roles": row.get("alt_roles_list", []),
        "crosswalk_confirmed": row.get("crosswalk_confirmed", False),

        # JUSTIFICATION (truncated)
        "grouping_justification": (row.get("grouping_justification") or "")[:1500],

        # SCORING BREAKDOWN
        "SCORE_CALCULATION": f"~{row.get('human_error_probability',0)/20:.1f} base + ({row.get('alt_count',0)} alts × 0.5) + ({row.get('confusion_risk_score',0)} confusion × 0.5) = {row.get('likelihood_error_score_0_5')} ({row.get('likelihood_band')} risk)"
    }

    user_prompt = json.dumps(payload, indent=2)

    for attempt in range(max_retries):
        try:
            # Call appropriate API based on provider
            if config["provider"] == "anthropic":
                message = client.messages.create(
                    model=config["model_id"],
                    max_tokens=config["max_tokens"],
                    temperature=config["temperature"],
                    system=JUDGE_SYSTEM,
                    messages=[{"role": "user", "content": user_prompt}]
                )
                response_text = message.content[0].text

            elif config["provider"] == "openai":
                response = client.chat.completions.create(
                    model=config["model_id"],
                    messages=[
                        {"role": "system", "content": JUDGE_SYSTEM},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=config["max_tokens"],
                    temperature=config["temperature"]
                )
                response_text = response.choices[0].message.content

            elif config["provider"] == "huggingface":
                completion = client.chat.completions.create(
                    model=config["model_id"],
                    messages=[
                        {"role": "system", "content": JUDGE_SYSTEM},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=config["max_tokens"],
                    temperature=config["temperature"]
                )
                response_text = completion.choices[0].message.content

            # Extract JSON from response
            json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', response_text, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group(0))

                # Validate required fields
                required = ["score_assessment", "confidence", "notes"]
                if all(k in result for k in required):
                    return result

            print(f"⚠️ Invalid JSON in response for {row.get('job_title_original')}")

        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "rate" in error_str.lower():
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"⚠️ Rate limited. Retrying in {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                    time.sleep(wait_time)
                    continue
            else:
                print(f"⚠️ Error on {row.get('job_title_original')}: {error_str[:100]}")
                break

    # Fallback response
    return {
        "hedging_language": False,
        "title_only_reasoning": False,
        "mentions_competing_role_terms": False,
        "score_assessment": "unknown",
        "confidence": "low",
        "notes": "API error or invalid response"
    }

print("✅ Judge system initialized with enhanced prompt")


In [ ]:
# ==== 13) Execute LLM Judge on Moderate+ Records ====

if len(moderate_or_higher) == 0:
    print("✅ No Moderate+ risk records found - skipping LLM evaluation")
    judged_df = pd.DataFrame()
else:
    print(f"🔍 Evaluating {len(moderate_or_higher)} Moderate+ records using {MODEL_CHOICE}...")
    print(f"📊 Distribution: {moderate_or_higher['likelihood_band'].value_counts().to_dict()}")
    print(f"\n⏱️ Estimated time: {len(moderate_or_higher) * 3} seconds\n")

    judged = []

    for idx, row in tqdm(moderate_or_higher.iterrows(), total=len(moderate_or_higher), desc=f"{MODEL_CHOICE} Review"):
        result = judge_record(row.to_dict())
        result["job_title_key"] = row["job_title_key"]
        result["source_row_index"] = row.get("source_row_index", idx)
        judged.append(result)

        # Rate limiting
        if config["provider"] == "anthropic":
            time.sleep(0.3)  # ~200 req/min allowed
        elif config["provider"] == "openai":
            time.sleep(0.1)  # Higher rate limit
        else:
            time.sleep(0.5)  # Conservative for HF

    judged_df = pd.DataFrame(judged)
    print(f"\n✅ {MODEL_CHOICE} evaluation complete!")

    # Quality checks
    print("\n📊 Score Assessment Distribution:")
    print(judged_df['score_assessment'].value_counts())

    print("\n📊 Confidence Distribution:")
    print(judged_df['confidence'].value_counts())

    # Flag potential calibration issues
    suspicious = judged_df[
        (judged_df['score_assessment'] == 'too_high') &
        (judged_df['confidence'] == 'high')
    ]

    if len(suspicious) > len(judged_df) * 0.4:  # More than 40% flagged
        print(f"\n⚠️ WARNING: {len(suspicious)}/{len(judged_df)} records flagged as confidently 'too_high'")
        print("   This may indicate judge miscalibration - review sample manually")
        print("   Consider switching to Claude Sonnet 4.5 for better accuracy")

    # Show sample results
    print("\n📋 Sample LLM Evaluations:")
    sample_cols = ["job_title_key", "likelihood_error_score_0_5", "score_assessment",
                   "confidence", "notes"]
    sample_cols = [c for c in sample_cols if c in judged_df.columns]
    display(judged_df[sample_cols].head(10))


In [ ]:
# ==== 14) Export results with LLM enhancements ====

import os
from google.colab import files

OUTDIR = "/content/output_likelihood_error"
os.makedirs(OUTDIR, exist_ok=True)

# Check if LLM results exist
has_llm_results = 'judged_df' in locals() and not judged_df.empty

if has_llm_results:
    print(f"🚀 Exporting Hybrid Results (Deterministic + {MODEL_CHOICE})\n")

    # Merge LLM results
    llm_cols = ["job_title_key", "hedging_language", "title_only_reasoning",
                "mentions_competing_role_terms", "score_assessment", "confidence", "notes"]
    available_llm_cols = [c for c in llm_cols if c in judged_df.columns]

    df_export = df.merge(judged_df[available_llm_cols], on="job_title_key", how="left")
    df_export["llm_reviewed"] = df_export["confidence"].notna()

    # Add model info
    df_export["llm_judge_model"] = ""
    df_export.loc[df_export["llm_reviewed"], "llm_judge_model"] = config["model_id"]
else:
    print("📊 Exporting Deterministic Results Only\n")
    df_export = df.copy()
    df_export["llm_reviewed"] = False
    df_export["llm_judge_model"] = ""

# Generate filename
model_suffix = MODEL_CHOICE.replace("-", "_").replace(".", "_") if has_llm_results else "deterministic"
csv_filename = f"Job_Classifications_Batch_with_Likelihood_Error_{model_suffix}.csv"
csv_path = os.path.join(OUTDIR, csv_filename)

# Export
df_export.to_csv(csv_path, index=False)
print(f"✅ Exported: {csv_filename}")

# Summary Report
print("-" * 60)
print("📊 SUMMARY REPORT")
print("-" * 60)
print(f"Total records processed:  {len(df_export)}")
print(f"LLM reviewed:             {df_export['llm_reviewed'].sum()}")
if has_llm_results:
    print(f"Judge model used:         {config['model_id']}")
print("\n📊 Risk Band Distribution:")
print(df_export["likelihood_band"].value_counts().sort_index())

if has_llm_results:
    print("\n📊 LLM Assessment Results:")
    reviewed = df_export[df_export["llm_reviewed"]]
    print(f"  Appropriate: {(reviewed['score_assessment'] == 'appropriate').sum()}")
    print(f"  Too high:    {(reviewed['score_assessment'] == 'too_high').sum()}")
    print(f"  Too low:     {(reviewed['score_assessment'] == 'too_low').sum()}")
    print(f"  Unknown:     {(reviewed['score_assessment'] == 'unknown').sum()}")

print("-" * 60)
print("\n⬇️ Downloading results...")
files.download(csv_path)
print("✅ Complete!")


---
## 🔍 Validation & Analysis (Optional)

Run these cells to analyze the quality of results and identify potential issues.


In [ ]:
# ==== Optional: Quality Analysis ====

if has_llm_results:
    print("🔍 QUALITY ANALYSIS\n")

    # Analysis 1: Score vs Assessment Alignment
    print("1️⃣ Score vs Assessment Alignment Check:")
    print("-" * 40)

    reviewed = df_export[df_export["llm_reviewed"]].copy()

    # High scores marked as "too_high"
    high_score_flagged = reviewed[
        (reviewed["likelihood_error_score_0_5"] >= 2.0) &
        (reviewed["score_assessment"] == "too_high") &
        (reviewed["alt_count"] >= 2)
    ]

    if len(high_score_flagged) > 0:
        print(f"⚠️ {len(high_score_flagged)} high scores flagged as 'too_high' despite multiple alternatives:")
        display(high_score_flagged[["job_title_key", "likelihood_error_score_0_5",
                                    "alt_count", "confusion_risk_score", "notes"]].head())
    else:
        print("✅ No suspicious 'too_high' flags")

    # Low scores marked as "too_low"
    print("\n2️⃣ Low Score Validation:")
    print("-" * 40)
    low_score_flagged = reviewed[
        (reviewed["likelihood_error_score_0_5"] < 2.0) &
        (reviewed["score_assessment"] == "too_low")
    ]

    if len(low_score_flagged) > 0:
        print(f"⚠️ {len(low_score_flagged)} low scores flagged as 'too_low':")
        display(low_score_flagged[["job_title_key", "likelihood_error_score_0_5",
                                   "alt_count", "confusion_risk_score", "notes"]].head())
    else:
        print("✅ No low scores flagged as 'too_low'")

    # Confidence distribution by assessment
    print("\n3️⃣ Confidence by Assessment Type:")
    print("-" * 40)
    conf_pivot = reviewed.groupby(["score_assessment", "confidence"]).size().unstack(fill_value=0)
    display(conf_pivot)

    # Hedging language detection
    print("\n4️⃣ Justification Quality Flags:")
    print("-" * 40)
    print(f"Hedging language detected: {reviewed['hedging_language'].sum()}")
    print(f"Title-only reasoning:      {reviewed['title_only_reasoning'].sum()}")
    print(f"Competing roles mentioned: {reviewed['mentions_competing_role_terms'].sum()}")

else:
    print("ℹ️ Run LLM evaluation first to see quality analysis")
